# Biophysical Reaction-Diffusion Transport and Evolutionary Sanctuary Dynamics in Compact EGFR-Mutant Lung Adenocarcinoma

**Author:** Luca Celebrano  
**Affiliation:** School of Medicine, University of Naples Federico II & TIGEM  
**DOI:** 10.5281/zenodo.23623653  

This notebook demonstrates the core computational models of the Master Study:
1. **Reaction-Diffusion Transport:** Exact Modified Bessel $ diffusion profiles and pharmacological sanctuary shells.
2. **Evolutionary Game Theory:** Hawk-Dove replicator dynamics and post-ADAURA cessation regrowth.
3. **Clinical Cohort Analysis:** Recurrence-free survival (RFS) across IASLC histological subtypes in the MSKCC 2020 cohort (=604$).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add src directory to path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from biophysical_diffusion_model import BiophysicalTransportModel
from replicator_dynamics_game import SpatialGameAssay
from clinical_validation import MSKCCClinicalValidator

## 1. Biophysical Reaction-Diffusion Transport (Bessel $ Solution)

In [ ]:
model = BiophysicalTransportModel()
sanctuary = model.compute_sanctuary_geometry(nest_radius_um=200.0)
print(f"Effective Diffusion Length: {model.lambda_um:.2f} um")
print(f"Critical Penetration Radius: {sanctuary['r_critical_um']:.2f} um")
print(f"Viable Sanctuary Shell Width: {sanctuary['viable_sanctuary_shell_width_um']:.2f} um")
print(f"Pharmacological Sanctuary Area: {sanctuary['viable_sanctuary_area_pct']:.2f}%")
print(f"Operational Regime: {sanctuary['operational_regime']}")

## 2. Evolutionary Game Theory & Replicator Dynamics

In [ ]:
game = SpatialGameAssay()
x_star = game.compute_nash_equilibrium()
stab = game.compute_lyapunov_stability()
bifurc = game.simulate_adaura_cessation_bifurcation(months_treatment=36, months_followup=24)

print(f"Polymorphic Nash Equilibrium: x* = {x_star:.4f}")
print(f"Lyapunov Stability Eigenvalue: lambda = {stab['lyapunov_eigenvalue']:.4f} (Asymptotically Stable)")
print(f"ctDNA Recurrence: Month {bifurc['mrd_recurrence_month']:.1f}")
print(f"CT Imaging Recurrence: Month {bifurc['ct_imaging_recurrence_month']:.1f}")
print(f"Diagnostic Lead Time: {bifurc['mrd_lead_time_months']:.1f} months")

## 3. MSKCC 2020 Clinical Cohort Validation (N = 604)

In [ ]:
data_file = Path.cwd().parent / "data" / "mskcc_2020_real_cohort.csv"
validator = MSKCCClinicalValidator(data_file)
summary = validator.get_cohort_summary()
print(f"Total Evaluated Patients: {summary['total_patients']}")
print(f"EGFR-Mutant Subpopulation: {summary['egfr_mutant_patients']}")
print(f"Total Observed Recurrences: {summary['total_recurrences']} ({summary['overall_recurrence_rate']*100:.1f}%)")

grad = validator.compute_subtype_gradient()
for subtype, metrics in grad.items():
    print(f"  - {subtype:15s}: N={metrics['N']:3d}, Recurrences={metrics['recurrences']:2d} ({metrics['recurrence_rate']*100:.1f}%)")

cox_res = validator.fit_multivariable_cox_models()
m1 = cox_res['model1_extreme']
print(f"
Architectural Hazard Ratio (Solid/Micro vs Lepidic): HR = {m1['adjusted_hr']:.2f} (95% CI: {m1['ci_lower']:.2f} - {m1['ci_upper']:.2f}, p = {m1['p_value']:.2e})")